# pKiss
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
import pandas as pd
import time
import subprocess
from pathlib import Path

In [ ]:
method_name = "pKiss"
base = Path.cwd()

# Set installation directory
install_dir = "../tools"
os.makedirs(install_dir, exist_ok=True)
os.chdir(install_dir)

In [ ]:
pkiss_env = "./pKiss"
if not os.path.exists(pkiss_env):
    print(f"Creating conda environment at: {pkiss_env}")
    !conda clean --index-cache -y
    result = subprocess.run(f"conda create --prefix '{pkiss_env}' python=3.8 -y", shell=True)
    if result.returncode != 0:
        raise RuntimeError("Failed to create conda environment")
    print("Conda environment created successfully")
else:
    print(f"pKiss environment already exists at {pkiss_env}")

In [ ]:
# Check whether pKiss is already usable in the environment
pkiss_check_cmd = ['conda', 'run', '--prefix', pkiss_env, 'pKiss', '--help']
try:
    result = subprocess.run(pkiss_check_cmd, capture_output=True, text=True, timeout=20)
    if result.returncode == 0:
        print("pKiss is already installed and usable")
    else:
        print("Installing pKiss into environment...")
        install_cmd = f"conda install --prefix '{pkiss_env}' -c bioconda pkiss -y"
        result = subprocess.run(install_cmd, shell=True)
        if result.returncode != 0:
            raise RuntimeError("Failed to install pKiss")
        print("pKiss installed successfully")
        result = subprocess.run(pkiss_check_cmd, capture_output=True, text=True, timeout=20)
        if result.returncode != 0:
            raise RuntimeError("pKiss still not usable after install")
except subprocess.TimeoutExpired:
    raise RuntimeError("Timed out while checking pKiss installation")

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
def run_folding(name: str, seq: str):
    tmp_fasta = f'pKiss_tmp_{name}.fasta'
    out_file_name = f'pKiss_clean_tmp_{name}.dot'

    with open(tmp_fasta, 'w') as ofile:
        ofile.write(f'>{name}\n{seq}\n')

    cmd = ['conda', 'run', '--prefix', pkiss_env, 'pKiss', '-mode', 'mfe', tmp_fasta]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        with open(out_file_name, 'w') as f:
            f.write(f'>{name}\n{seq}\nFAILED\n')
        os.remove(tmp_fasta)
        return out_file_name

    lines = result.stdout.strip().split('\n')
    structure = 'NO_STRUCTURE'
    if len(lines) >= 2:
        structure_line = lines[-1]
        if structure_line and not structure_line.startswith('>'):
            parts = structure_line.split()
            if len(parts) >= 2:
                structure = parts[-1]

    with open(out_file_name, 'w') as f:
        f.write(f'>{name}\n{seq}\n{structure}\n')

    os.remove(tmp_fasta)
    return out_file_name

In [ ]:
out_fasta_name = '../prediction/pKiss.fasta'
os.makedirs('../prediction', exist_ok=True)
if os.path.exists(out_fasta_name):
    os.remove(out_fasta_name)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}\t{'status'}")
successful_runs = 0

for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    dot_file_name = run_folding(vid, seq)

    status = 'SUCCESS'
    if os.path.exists(dot_file_name):
        with open(dot_file_name, 'r') as f:
            content = f.read().strip()
            if 'FAILED' in content or not content or len(content.split('\n')) < 3:
                status = 'FAILED'
            else:
                successful_runs += 1
    else:
        status = 'FAILED'

    if os.path.exists(dot_file_name):
        with open(dot_file_name, 'r') as f:
            content = f.read()
        with open(out_fasta_name, 'a') as f:
            f.write(content)

    elapsed_time = time.time() - start_time
    print(f"{elapsed_time: .1f} s\t{status}")

    if os.path.exists(dot_file_name):
        os.remove(dot_file_name)

print(f"\nProcessing complete. Results saved to {out_fasta_name}")